In [2]:
# =========================================================
# 0. Setup
# =========================================================
!pip -q install tensorflow tensorflow-datasets matplotlib
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

print("TF version:", tf.__version__)
device = "GPU" if tf.config.list_physical_devices("GPU") else "CPU"
print("Running on", device)
# ------------------------------------------------------------
# 1. Load MNIST the TF-2 way (32 000 train / 4 000 val / 4 000 test)
# ------------------------------------------------------------
def to_float(image, label):          # <-- names changed for clarity
    return tf.cast(image, tf.float32) / 255., label

train, info = tfds.load("mnist",
                        split="train[:90%]",
                        as_supervised=True,   # <-- gives (img, label) directly
                        with_info=True)
val, test = tfds.load("mnist",
                      split=["train[90%:]", "test"],
                      as_supervised=True)    # <-- same here

BATCH = 128
train = train.map(to_float).shuffle(5000).batch(BATCH).prefetch(tf.data.AUTOTUNE)
val   = val.map(to_float).batch(BATCH).prefetch(tf.data.AUTOTUNE)
test  = test.map(to_float).batch(BATCH).prefetch(tf.data.AUTOTUNE)

img_shape = info.features["image"].shape
classes   = info.features["label"].num_classes

TF version: 2.19.0
Running on CPU


In [3]:
# ------------------------------------------------------------
# 2. Helper: compute output size after conv/pool layers
#    (reproduces the formulas in the book)
# ------------------------------------------------------------
def out_size(i, k, s=1, padding="same"):
    if padding == "same":
        return int(np.ceil(i / s))
    else:  # valid
        return int(np.floor((i - k + s) / s))

# quick sanity check with the book example
print("same  :", out_size(70, 7, 1, "same"))   # 70
print("valid :", out_size(70, 7, 1, "valid"))  # 64

same  : 70
valid : 64


In [4]:
# ------------------------------------------------------------
# 3. Baseline LeNet-style CNN (matches section in book)
# ------------------------------------------------------------
DefaultConv = partial(tf.keras.layers.Conv2D,
                      kernel_size=3,
                      padding="same",
                      activation="relu",
                      kernel_initializer="he_normal")

baseline = tf.keras.Sequential([
    tf.keras.layers.Input(shape=img_shape),
    DefaultConv(filters=6,  kernel_size=5),
    tf.keras.layers.AveragePooling2D(pool_size=2),
    DefaultConv(filters=16, kernel_size=5),
    tf.keras.layers.AveragePooling2D(pool_size=2),
    DefaultConv(filters=120, kernel_size=5),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(84, activation="relu"),
    tf.keras.layers.Dense(classes, activation="softmax")
])

baseline.compile(optimizer="adam",
                 loss="sparse_categorical_crossentropy",
                 metrics=["accuracy"])

baseline.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 14, 14, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 7, 7, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 7, 7, 120)      │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 5880)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 84)             │       494,004 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 545,546 (2.08 MB)

 Trainable params: 545,546 (2.08 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# ------------------------------------------------------------
# 4. Train the baseline
# ------------------------------------------------------------
baseline_history = baseline.fit(train,
                                validation_data=val,
                                epochs=5)

Epoch 1/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 63s 143ms/step - accuracy: 0.8603 - loss: 0.4513 - val_accuracy: 0.9765 - val_loss: 0.0794
Epoch 2/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 55s 130ms/step - accuracy: 0.9799 - loss: 0.0667 - val_accuracy: 0.9842 - val_loss: 0.0518
Epoch 3/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 83s 132ms/step - accuracy: 0.9868 - loss: 0.0433 - val_accuracy: 0.9835 - val_loss: 0.0517
Epoch 4/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 81s 129ms/step - accuracy: 0.9892 - loss: 0.0341 - val_accuracy: 0.9863 - val_loss: 0.0453
Epoch 5/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 56s 132ms/step - accuracy: 0.9915 - loss: 0.0283 - val_accuracy: 0.9887 - val_loss: 0.0399


In [ ]:
# ------------------------------------------------------------
# 5. Modernized version: BatchNorm, Dropout, data augmentation
# ------------------------------------------------------------
data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(-0.1, 0.1),
])

modern = tf.keras.Sequential([
    tf.keras.layers.Input(shape=img_shape),
    data_aug,
    DefaultConv(64, kernel_size=3),
    tf.keras.layers.BatchNormalization(),
    DefaultConv(64, kernel_size=3),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPool2D(),
    tf.keras.layers.Dropout(0.25),

    DefaultConv(128, kernel_size=3),
    tf.keras.layers.BatchNormalization(),
    DefaultConv(128, kernel_size=3),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPool2D(),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(classes, activation="softmax")
])

modern.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
               loss="sparse_categorical_crossentropy",
               metrics=["accuracy"])

modern_history = modern.fit(train,
                            validation_data=val,
                            epochs=8)

Epoch 1/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 672s 2s/step - accuracy: 0.8875 - loss: 0.3756 - val_accuracy: 0.9813 - val_loss: 0.0543
Epoch 2/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 663s 2s/step - accuracy: 0.9751 - loss: 0.0832 - val_accuracy: 0.9892 - val_loss: 0.0353
Epoch 3/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 658s 2s/step - accuracy: 0.9818 - loss: 0.0596 - val_accuracy: 0.9907 - val_loss: 0.0325
Epoch 4/8
241/422 ━━━━━━━━━━━━━━━━━━━━ 4:35 2s/step - accuracy: 0.9849 - loss: 0.0465

In [ ]:
# ------------------------------------------------------------
# 6. Mini-ResNet: residual block with skip connection
#    (illustrates the core ResNet idea from the chapter)
# ------------------------------------------------------------
class ResidualUnit(tf.keras.layers.Layer):
    def __init__(self, filters, strides=1, **kw):
        super().__init__(**kw)
        self.filters   = filters
        self.strides   = strides
        self.main_path = [
            DefaultConv(filters,   strides=strides),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.ReLU(),
            DefaultConv(filters),
            tf.keras.layers.BatchNormalization()
        ]
        self.skip_path = []
        if strides > 1:
            self.skip_path = [
                tf.keras.layers.Conv2D(filters, 1, strides=strides,
                                       padding="same"),
                tf.keras.layers.BatchNormalization()
            ]

    def call(self, X):
        Z = X
        for layer in self.main_path:
            Z = layer(Z)
        skip_Z = X
        for layer in self.skip_path:
            skip_Z = layer(skip_Z)
        return tf.keras.activations.relu(Z + skip_Z)

    def get_config(self):
        return {"filters": self.filters, "strides": self.strides}

# Build a tiny ResNet-8
resnet = tf.keras.Sequential([
    tf.keras.layers.Input(shape=img_shape),
    DefaultConv(64, kernel_size=3),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    ResidualUnit(64),
    ResidualUnit(64),
    ResidualUnit(128, strides=2),  # down-sample
    ResidualUnit(128),
    ResidualUnit(256, strides=2),  # down-sample
    ResidualUnit(256),

    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(classes, activation="softmax")
])

resnet.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
               loss="sparse_categorical_crossentropy",
               metrics=["accuracy"])

resnet_history = resnet.fit(train,
                            validation_data=val,
                            epochs=8)

In [ ]:
# ------------------------------------------------------------
# 7. Evaluate all three models on the held-out test set
# ------------------------------------------------------------
for name, model in zip(["baseline", "modern", "resnet"],
                       [baseline, modern, resnet]):
    loss, acc = model.evaluate(test, verbose=0)
    print(f"{name:8s} – test accuracy: {acc:.4f}")

In [ ]:
# ------------------------------------------------------------
# 8. Visualize learned first-layer filters (modern model)
# ------------------------------------------------------------
first_conv = modern.layers[2]  # after aug, first conv
filters    = first_conv.get_weights()[0]  # shape (3,3,1,64)
filters    = filters[..., 0, :]            # grayscale

n_show = 32
fig, ax = plt.subplots(4, 8, figsize=(12, 6))
for i in range(n_show):
    ax.flat[i].imshow(filters[:, :, i], cmap="viridis")
    ax.flat[i].axis("off")
plt.suptitle("First-layer convolutional kernels")
plt.show()

In [ ]:
# ------------------------------------------------------------
# 9. Memory footprint estimation (one forward pass)
# ------------------------------------------------------------
# helper: count trainable parameters
def param_size(model):
    return np.sum([np.prod(v.shape) for v in model.trainable_weights])

print("Trainable params:")
for name, model in zip(["baseline", "modern", "resnet"],
                       [baseline, modern, resnet]):
    print(f"{name:8s}: {param_size(model):,}")